# Tugas 2: Transformasi Gambar, Image Compression, dan Watermarking
**Mata Kuliah:** Komputasi Multimedia  
**Topik:** Transformasi Citra (Augmentasi Data), Kompresi Citra Digital, dan Watermarking Citra  
**Format Pengerjaan:** Individu / Kelompok (Maksimal 3 Orang)  
**Deliverables:** Laporan PDF + Notebook Python (.ipynb) + Folder Data Hasil

---

### Pernyataan Orisinalitas & Penggunaan AI (Academic Integrity)
*Seluruh pekerjaan yang disajikan dalam dokumen ini telah dipahami secara mendalam dan diverifikasi kebenarannya oleh mahasiswa. Bantuan AI (Google Antigravity / DeepMind Assistant) digunakan sebagai mitra penulisan kode, kurasi literatur ilmiah, dan automasi pemodelan visual. Seluruh formulasi matematis, analisis komparatif, dan eksekusi kode telah diuji secara empiris dan dapat direproduksi secara utuh.*


## 0. Persiapan Lingkungan dan Instalasi Dependensi
Sebelum menjalankan notebook, pastikan pustaka-pustaka ilmiah berikut telah terpasang:
```bash
pip install numpy pillow matplotlib scipy scikit-image opencv-python reportlab svglib pandas
```


In [ ]:
# Import modul standar dan pustaka ilmiah
import os
import sys
import time
import math
import random
import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import matplotlib.pyplot as plt
import pandas as pd
from scipy.fftpack import dct, idct
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim
from IPython.display import display

# Inisialisasi Random Seed untuk Reproduksibilitas Eksperimen
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"[STATUS] Random seed diatur secara global ke: {RANDOM_SEED}")

# Setup Direktori Data dan Output
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

DIR_VIDEO = os.path.join(DATA_DIR, "video")
DIR_EXTRACTED = os.path.join(DATA_DIR, "extracted_frames")
DIR_SAMPLES = os.path.join(DATA_DIR, "sample_10_frames")
DIR_AUGMENTED = os.path.join(DATA_DIR, "augmented_images")
DIR_COMP_INPUTS = os.path.join(DATA_DIR, "compression_inputs")
DIR_COMP_RESULTS = os.path.join(DATA_DIR, "compression_results")
DIR_WATERMARK = os.path.join(DATA_DIR, "watermark_results")

for d in [DIR_VIDEO, DIR_EXTRACTED, DIR_SAMPLES, DIR_AUGMENTED, DIR_COMP_INPUTS, DIR_COMP_RESULTS, DIR_WATERMARK, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print("[STATUS] Struktur folder direktori terverifikasi.")


---
## Bagian A. Augmentasi Data pada Dataset DAiSEE

### 1. Landasan Teori dan Konteks Dataset DAiSEE
Dataset **DAiSEE (*Dataset for Affective States in E-learning Environments*)** (Kumar et al., 2016) merupakan dataset patokan (*benchmark*) untuk menganalisis keadaan afektif siswa dalam pembelajaran daring *in-the-wild*. DAiSEE mencakup 4 label afektif utama:
1. **Engagement** (Keterlibatan)
2. **Boredom** (Kebosanan)
3. **Confusion** (Kebingungan)
4. **Frustration** (Frustrasi)

Video direkam dari kamera web (*webcam*) laptop/komputer siswa di lingkungan belajar alami dengan berbagai tantangan visual: orientasi kepala tidak konstan, pencahayaan ruangan yang dinamis, derau sensor kamera murah, dan fluktuasi jarak siswa ke layar.

### 2. Penjelasan 6 Teknik Augmentasi Data & Justifikasi Kesesuaiannya

1. **Horizontal Flipping (Pembalikan Horizontal)**
   - *Cara Kerja:* Membalik matriks citra secara horizontal melalui sumbu $y$: $I_{aug}(x, y) = I(W - 1 - x, y)$.
   - *Justifikasi untuk DAiSEE:* **SANGAT SESUAI DAN AMAN.** Wajah manusia memiliki simetri bilateral alami. Ekspresi keterlibatan, kebingungan (mengerutkan alis / AU4), dan kebosanan muncul secara simetris di kedua sisi wajah. Pembalikan horizontal melipatgandakan data tanpa mengubah makna semantik keadaan afektif (*label-preserving*) (Shorten & Khoshgoftaar, 2019).
   - *Batasan:* Jika terdapat mikrogelisah asimetris, flip tetap valid untuk memvariasikan arah tatapan siswa (menoleh ke sisi kiri vs kanan layar).

2. **Slight Rotation (Rotasi Sudut Kecil: $\\pm 8^\\circ$)**
   - *Cara Kerja:* Transformasi afina matriks rotasi 2D terhadap titik pusat citra $(c_x, c_y)$ dengan interpolasi bilinear dan *reflection border padding*.
   - *Justifikasi untuk DAiSEE:* **SANGAT SESUAI DENGAN SUDUT KECIL ($\\le 10^\\circ$), NAMUN ROTASI BESAR ($>30^\\circ$ ATAU $90^\\circ$) TIDAK SESUAI.**
     - Siswa dalam pembelajaran daring duduk tegak di depan meja/layar laptop di bawah pengaruh gravitasi bumi.
     - Kepala siswa tidak pernah berputar $90^\\circ$ atau terbalik $180^\\circ$. Rotasi besar merusak *natural head-pose prior* dan menurunkan performa detektor wajah/pose (Kumar et al., 2016; Dewan et al., 2019).
     - Sebaliknya, rotasi kecil ($\pm 5^\circ - 10^\circ$) sangat natural untuk mereplikasi *head tilt* ketika siswa sedang berpikir, bingung, atau mencatat.

3. **Random Cropping & Scaling (Pemotongan & Penskalaan Ulang: Rasio 0.88)**
   - *Cara Kerja:* Memotong area sub-persegi panjang sebesar 88% di sekitar pusat wajah, lalu mereskalanya kembali ke resolusi asal menggunakan interpolasi bilinear.
   - *Justifikasi untuk DAiSEE:* **SANGAT SESUAI.** Mensimulasikan pergerakan tubuh siswa yang condong maju ke arah layar saat sangat berkonsentrasi (*high engagement*) atau menyandarkan punggung ke kursi saat lelah (*boredom*).

4. **Color Jitter / Brightness & Contrast Adjustment ($\\alpha = 1.18, \\beta = 15$)**
   - *Cara Kerja:* Transformasi intensitas linier $I_{aug}(x, y) = \\text{clip}(\\alpha \\cdot I(x, y) + \\beta, 0, 255)$, di mana $\\alpha$ mengatur kontras dan $\\beta$ mengatur kecerahan.
   - *Justifikasi untuk DAiSEE:* **SANGAT SESUAI.** Video DAiSEE direkam di ruangan dengan kondisi cahaya liar (*in-the-wild*): cahaya matahari yang berubah sepanjang hari, lampu meja, serta pantulan cahaya layar monitor yang berubah dinamis saat materi video berubah warna.

5. **Gaussian Noise Injection ($\\mu = 0, \\sigma = 15$)**
   - *Cara Kerja:* Menambahkan nilai derau acak berdistribusi normal $\\mathcal{N}(0, \\sigma^2)$ pada setiap saluran warna piksel.
   - *Justifikasi untuk DAiSEE:* **SANGAT SESUAI.** Kamera web berbiaya rendah memiliki sensor CMOS kecil dengan sensitivitas rendah dalam kondisi pencahayaan redup (*low-light noise*), memicu derau termal dan derau kuantisasi.

6. **Gaussian Blur ($k = 5\\times 5, \\sigma = 1.2$)**
   - *Cara Kerja:* Konvolusi citra spasial dengan kernel Gaussian 2D berbobot jarak radial $G(x, y) = \\frac{1}{2\\pi\\sigma^2} e^{-\\frac{x^2+y^2}{2\\sigma^2}}$.
   - *Justifikasi untuk DAiSEE:* **SANGAT SESUAI.** Mensimulasikan *defocus* kamera web murah serta *motion blur* yang terjadi ketika siswa mengangguk atau menggelengkan kepala dengan cepat.

### 3. Referensi Ilmiah Pendukung
1. **Kumar, A., Sharma, R., & Vig, L. (2016).** *DAiSEE: Towards User Engagement Recognition in the Wild.* arXiv preprint arXiv:1609.01883. DOI: [10.48550/arXiv.1609.01883](https://doi.org/10.48550/arXiv.1609.01883).
2. **Shorten, C., & Khoshgoftaar, T. M. (2019).** *A survey on Image Data Augmentation for Deep Learning.* Journal of Big Data, 6(1), 60. DOI: [10.1186/s40537-019-0197-0](https://doi.org/10.1186/s40537-019-0197-0).
3. **Dewan, M. A. A., Murshed, M., & Lin, F. (2019).** *Engagement analysis in online learning: A review.* Computers & Education, 141, 103628. DOI: [10.1016/j.compedu.2019.103628](https://doi.org/10.1016/j.compedu.2019.103628).


In [ ]:
# 1. Ekstraksi Frame Video Asli DAiSEE
def find_daisee_video(video_dir):
    daisee_base = os.path.join(video_dir, "DAiSEE")
    if os.path.exists(daisee_base):
        for root, dirs, files in os.walk(daisee_base):
            for f in files:
                if f.lower().endswith(('.avi', '.mp4')):
                    return os.path.join(root, f)
    return os.path.join(video_dir, "daisee_student_engagement.mp4")

video_file = find_daisee_video(DIR_VIDEO)
print(f"[INFO] Membaca video DAiSEE: {video_file}")

cap = cv2.VideoCapture(video_file)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
print(f"[INFO] Total frame: {total_frames} | FPS: {fps:.1f}")

extracted_frame_paths = []
frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    save_path = os.path.join(DIR_EXTRACTED, f"frame_{frame_idx:04d}.png")
    cv2.imwrite(save_path, frame)
    extracted_frame_paths.append(save_path)
    frame_idx += 1
cap.release()

print(f"[SUKSES] Berhasil mengekstrak {len(extracted_frame_paths)} frame ke {DIR_EXTRACTED}")

# 2. Pengambilan 10 Gambar Secara Acak (Reproducible Seed)
random.seed(RANDOM_SEED)
sampled_10_paths = sorted(random.sample(extracted_frame_paths, 10))

print(f"\n[INFO] 10 Gambar Terpilih Acak (Seed = {RANDOM_SEED}):")
for i, path in enumerate(sampled_10_paths, 1):
    dest = os.path.join(DIR_SAMPLES, f"sample_{i:02d}_{os.path.basename(path)}")
    img = cv2.imread(path)
    cv2.imwrite(dest, img)
    print(f"  {i:02d}. {os.path.basename(dest)}")


In [ ]:
# 3. Definisi Fungsi Augmentasi Citra
def augment_horizontal_flip(image):
    return cv2.flip(image, 1)

def augment_slight_rotation(image, angle=8.0):
    h, w = image.shape[:2]
    center = (w // 2, h // 2)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(image, matrix, (w, h), borderMode=cv2.BORDER_REFLECT_101)

def augment_random_crop_scale(image, crop_ratio=0.88):
    h, w = image.shape[:2]
    ch, cw = int(h * crop_ratio), int(w * crop_ratio)
    top = int((h - ch) * 0.45)
    left = int((w - cw) * 0.5)
    cropped = image[top:top+ch, left:left+cw]
    return cv2.resize(cropped, (w, h), interpolation=cv2.INTER_LINEAR)

def augment_brightness_contrast(image, alpha=1.18, beta=15):
    return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

def augment_gaussian_noise(image, mean=0, sigma=15):
    gauss = np.random.normal(mean, sigma, image.shape).astype(np.float32)
    noisy = np.clip(image.astype(np.float32) + gauss, 0, 255).astype(np.uint8)
    return noisy

def augment_gaussian_blur(image, ksize=(5, 5), sigma=1.2):
    return cv2.GaussianBlur(image, ksize, sigma)

AUG_MAP = {
    "1_horizontal_flip": augment_horizontal_flip,
    "2_slight_rotation": lambda img: augment_slight_rotation(img, angle=8.0),
    "3_random_crop_scale": lambda img: augment_random_crop_scale(img, crop_ratio=0.88),
    "4_color_jitter_brightness": lambda img: augment_brightness_contrast(img, alpha=1.18, beta=15),
    "5_gaussian_noise": lambda img: augment_gaussian_noise(img, sigma=15),
    "6_gaussian_blur": lambda img: augment_gaussian_blur(img, ksize=(5, 5), sigma=1.2)
}

# Terapkan pada seluruh 10 sampel dan simpan ke folder terpisah
for tech_name, func in AUG_MAP.items():
    folder = os.path.join(DIR_AUGMENTED, tech_name)
    os.makedirs(folder, exist_ok=True)
    for p in sampled_10_paths:
        fname = os.path.basename(p)
        raw_bgr = cv2.imread(p)
        aug_bgr = func(raw_bgr)
        cv2.imwrite(os.path.join(folder, f"aug_{fname}"), aug_bgr)

print("[SUKSES] Seluruh hasil augmentasi tersimpan di folder masing-masing.")


In [ ]:
# 4. Visualisasi Grid Berdampingan (Citra Asli vs Hasil Augmentasi)
grid_fig_path = os.path.join(OUTPUT_DIR, "augmentation_grid_comparison.png")
if os.path.exists(grid_fig_path):
    grid_img = Image.open(grid_fig_path)
    plt.figure(figsize=(16, 9))
    plt.imshow(grid_img)
    plt.axis('off')
    plt.title("Visualisasi Grid Komparatif Augmentasi Citra DAiSEE", fontsize=14, fontweight='bold', pad=15)
    plt.show()


---
## Bagian B. Image Compression (Kompresi Citra)

### 1. Rangkuman 10 Teknik Kompresi Citra Digital

1. **Run-Length Encoding (RLE)**
   - *Algoritma:* Mengganti deretan simbol identik berurutan (*runs*) menjadi pasangan `(panjang_run, nilai_simbol)`.
   - *Kelebihan:* Sangat cepat, kompleksitas $O(N)$, implementasi sederhana tanpa *memory overhead*.
   - *Kekurangan:* Rasio kompresi sangat rendah bahkan ekspansi data (negatif) pada citra fotografi kontinu dengan variasi frekuensi tinggi.
   - *Sitasi:* Golomb, S. (1966). *Run-length encodings.* IEEE Trans. Inform. Theory. DOI: [10.1109/TIT.1966.1053907](https://doi.org/10.1109/TIT.1966.1053907).

2. **Huffman Coding**
   - *Algoritma:* *Entropy coding* berbasis pohon biner prefiks optimal. Simbol dengan frekuensi kemunculan tinggi diberi kode bit pendek, sedangkan simbol jarang diberi bit panjang.
   - *Kelebihan:* Optimal untuk kode prefiks berbasis simbol tunggal, *lossless*, decoding deterministik cepat.
   - *Kekurangan:* Panjang kode bilangan bulat per simbol membatasi efisiensi bila probabilitas simbol bukan kelipatan $2^{-k}$.
   - *Sitasi:* Huffman, D. A. (1952). *A Method for the Construction of Minimum-Redundancy Codes.* Proc. IRE. DOI: [10.1109/JRPROC.1952.273898](https://doi.org/10.1109/JRPROC.1952.273898).

3. **Arithmetic Coding**
   - *Algoritma:* Memetakan seluruh deretan pesan ke sub-interval bilangan riil tunggal $[0, 1)$ secara bertahap berdasarkan probabilitas kumulatif simbol.
   - *Kelebihan:* Mencapai batas entropi teoritis Shannon (panjang bit fraksional), efisiensi kompresi lebih tinggi dari Huffman.
   - *Kekurangan:* Kompleksitas komputasi floating-point tinggi, rentan *error propagation*, paten historis membatasi adopsi awal.
   - *Sitasi:* Witten, I. H., Neal, R. M., & Cleary, J. G. (1987). *Arithmetic coding for data compression.* Commun. ACM. DOI: [10.1145/214762.214771](https://doi.org/10.1145/214762.214771).

4. **Lempel-Ziv-Welch (LZW)**
   - *Algoritma:* Kompresi berbasis kamus dinamis (*dictionary-based*). Mengganti deretan substring berulang dengan indeks kamus yang dibangun *on-the-fly*.
   - *Kelebihan:* Tidak memerlukan transmisi kamus eksplisit, efektif untuk teks dan grafis komputer berpola (basis format GIF dan TIFF).
   - *Kekurangan:* Tidak efisien untuk foto alami dengan variasi piksel unik, konsumsi memori kamus bertambah besar.
   - *Sitasi:* Welch, T. A. (1984). *A Technique for High-Performance Data Compression.* Computer. DOI: [10.1109/MC.1984.1659158](https://doi.org/10.1109/MC.1984.1659158).

5. **DEFLATE (LZ77 + Huffman)**
   - *Algoritma:* Standar kompresi format PNG. Menggunakan *sliding window* LZ77 untuk mencari duplikasi byte (panjang & jarak), dilanjutkan dengan pengkodean entropi pohon Huffman ganda.
   - *Kelebihan:* *Lossless*, rasio kompresi sangat baik pada grafis garis/web, didukung secara universal bebas royalti.
   - *Kekurangan:* Enkoder membutuhkan penelusuran *sliding window* intensif (kompleksitas kompresi lambat).
   - *Sitasi:* Deutsch, P. (1996). *DEFLATE Compressed Data Format Specification version 1.3.* IETF RFC 1951. DOI: [10.17487/RFC1951](https://doi.org/10.17487/RFC1951).

6. **JPEG (Discrete Cosine Transform - DCT)**
   - *Algoritma:* Transformasi warna YCbCr $\rightarrow$ *Chroma Subsampling* $\rightarrow$ Partisi blok $8\times 8$ $\rightarrow$ 2D-DCT $\rightarrow$ Kuantisasi psikovisual $\rightarrow$ Pemindaian Zigzag $\rightarrow$ Huffman/RLE.
   - *Kelebihan:* Rasio kompresi tinggi (10:1 hingga 30:1) dengan kualitas visual baik untuk citra fotografi alami.
   - *Kekurangan:* Menghasilkan *blocking artifacts* dan *ringing artifacts* pada rasio tinggi, tidak sesuai untuk teks atau grafis garis tajam.
   - *Sitasi:* Wallace, G. K. (1991). *The JPEG still picture compression standard.* IEEE Trans. Consum. Electron. DOI: [10.1109/30.89988](https://doi.org/10.1109/30.89988).

7. **JPEG 2000 (Discrete Wavelet Transform - DWT)**
   - *Algoritma:* Transformasi Wavelet Multi-resolusi (biorthogonal 9/7 lossy atau 5/3 lossless) $\rightarrow$ Kuantisasi skalar $\rightarrow$ EBCOT (*Embedded Block Coding with Optimal Truncation*).
   - *Kelebihan:* Eliminasi *blocking artifacts*, transmisi progresif (SNR dan resolusi), mendukung ROI (*Region of Interest*) coding.
   - *Kekurangan:* Kompleksitas komputasi sangat tinggi, konsumsi memori besar, adopsi peramban web terbatas.
   - *Sitasi:* Skodras, A., Christopoulos, C., & Ebrahimi, T. (2001). *The JPEG 2000 still image compression standard.* IEEE Signal Process. Mag. DOI: [10.1109/79.952801](https://doi.org/10.1109/79.952801).

8. **WebP (VP8/VP8L Intra-Prediction + Arithmetic/LZ77)**
   - *Algoritma:* Mengadaptasi prediksi intra-frame dari video VP8. Untuk mode *lossy*: prediksi spasial blok tetangga $\rightarrow$ residu di-DCT $\rightarrow$ kuantisasi $\rightarrow$ pengkodean aritmatika. Untuk *lossless*: transformasi spasial/warna $\rightarrow$ LZ77 + Huffman.
   - *Kelebihan:* Menghasilkan ukuran file 25-34% lebih kecil dari JPEG pada kualitas SSIM setara, mendukung transparansi alpha pada lossy dan lossless.
   - *Kekurangan:* Waktu enkoding lebih lambat dari JPEG standar pada perangkat keras tanpa akselerasi khusus.
   - *Sitasi:* Google Developers. (2020). *WebP Compression Study and Specification.* WebP Documentation.

9. **Vector Quantization (VQ)**
   - *Algoritma:* Mengelompokkan blok vektor piksel ($k$ dimensi) ke dalam vektor representatif terdekat dalam sebuah buku kode (*codebook*) menggunakan algoritma LBG/K-Means. Hanya indeks buku kode yang ditransmisikan.
   - *Kelebihan:* Dekoder sangat cepat (hanya pembacaan tabel/indeks), rasio kompresi tinggi.
   - *Kekurangan:* Pembuatan *codebook* pada tahap enkoding sangat mahal ($O(N \cdot K)$), degradasi visual tinggi (*edge blurring*).
   - *Sitasi:* Gray, R. (1984). *Vector quantization.* IEEE ASSP Magazine. DOI: [10.1109/MASSP.1984.1162229](https://doi.org/10.1109/MASSP.1984.1162229).

10. **Fractal Image Compression**
    - *Algoritma:* Berbasis *Partitioned Iterated Function Systems* (PIFS). Memanfaatkan *self-similarity* citra pada skala berbeda dengan memetakan blok domain besar ke blok rentang (*range block*) kecil melalui transformasi afina kontraktif.
    - *Kelebihan:* *Resolution independent* (dapat didekompresi ke resolusi lebih tinggi tanpa pikselasi tajam), dekoder cepat.
    - *Kekurangan:* Waktu pencarian blok domain saat enkoding luar biasa lambat, kompleksitas asimetris ekstrem.
    - *Sitasi:* Jacquin, A. E. (1992). *Image coding based on a fractal theory of iterated contractive image transformations.* IEEE Trans. Image Process. DOI: [10.1109/5.136869](https://doi.org/10.1109/5.136869).

---

### 2. Tabel Perbandingan 10 Teknik Kompresi Citra Digital

| No | Teknik Kompresi | Jenis | Mekanisme Utama | Rasio Kompresi Tipikal | Kualitas Hasil | Kompleksitas Komputasi | Contoh Format / Penggunaan | Referensi Utama (DOI) |
|---|---|---|---|---|---|---|---|---|
| 1 | **RLE** | Lossless | Run piksel sekuensial | 1:1 - 3:1 (grafis) / <1:1 (foto) | Identik sempurna | Enkoder: Rendah, Dekoder: Rendah | BMP, TIFF, Fax Group 3 | [10.1109/TIT.1966.1053907](https://doi.org/10.1109/TIT.1966.1053907) |
| 2 | **Huffman** | Lossless | Pohon biner frekuensi | 1.5:1 - 2.5:1 | Identik sempurna | Enkoder: Sedang, Dekoder: Rendah | Tahap akhir JPEG, GZIP | [10.1109/JRPROC.1952.273898](https://doi.org/10.1109/JRPROC.1952.273898) |
| 3 | **Arithmetic** | Lossless | Pemetaan interval $[0, 1)$ | 1.8:1 - 3:1 | Identik sempurna | Enkoder: Tinggi, Dekoder: Tinggi | JBIG, JPEG 2000, HEVC | [10.1145/214762.214771](https://doi.org/10.1145/214762.214771) |
| 4 | **LZW** | Lossless | Kamus dinamis string | 2:1 - 4:1 (grafis) | Identik sempurna | Enkoder: Sedang, Dekoder: Rendah | GIF, TIFF, PDF | [10.1109/MC.1984.1659158](https://doi.org/10.1109/MC.1984.1659158) |
| 5 | **DEFLATE** | Lossless | LZ77 sliding window + Huffman | 2:1 - 5:1 | Identik sempurna | Enkoder: Sedang-Tinggi, Dekoder: Rendah | PNG, ZIP, Web Assets | [10.17487/RFC1951](https://doi.org/10.17487/RFC1951) |
| 6 | **JPEG** | Lossy | 2D-DCT $8\times 8$ + Kuantisasi | 10:1 - 30:1 | Baik pada rasio sedang; *blocking* pada rasio tinggi | Enkoder: Sedang, Dekoder: Sedang | JPEG/JFIF, Kamera Digital | [10.1109/30.89988](https://doi.org/10.1109/30.89988) |
| 7 | **JPEG 2000** | Lossless / Lossy | DWT Wavelet + EBCOT | 20:1 - 50:1 (lossy) / 2:1 (lossless) | Sangat tinggi, tanpa *blocking* | Enkoder: Sangat Tinggi, Dekoder: Tinggi | Citra Medis DICOM, Sinema DCI | [10.1109/79.952801](https://doi.org/10.1109/79.952801) |
| 8 | **WebP** | Lossless / Lossy | VP8 Intra-prediction + DCT/LZ77 | 15:1 - 40:1 | Sangat tinggi, artefak halus | Enkoder: Tinggi, Dekoder: Sedang | Citra Web Modern, Android | Google WebP Docs |
| 9 | **Vector Quantization** | Lossy | Clustering Codebook LBG | 10:1 - 30:1 | Sedang, kabur pada batas tepi | Enkoder: Sangat Tinggi, Dekoder: Sangat Rendah | Kompresi Video Awal, VQ Speech | [10.1109/MASSP.1984.1162229](https://doi.org/10.1109/MASSP.1984.1162229) |
| 10 | **Fractal** | Lossy | PIFS Contractive Mapping | 20:1 - 80:1 | *Resolution-independent* | Enkoder: Ekstrem Tinggi, Dekoder: Rendah | Arsip Citra Resolusi Bebas | [10.1109/5.136869](https://doi.org/10.1109/5.136869) |

---

### 3. Penjelasan Perlakuan dan Konversi Format Vektor SVG
Format **SVG (*Scalable Vector Graphics*)** merupakan format grafis berbasis teks XML yang mendefinisikan objek citra secara matematis (titik koordinat, kurva Bezier parametrik, poligon, dan atribut gaya) alih-alih matriks piksel diskret.

Oleh karena itu:
- Algoritma kompresi citra raster (seperti DCT JPEG, kuantisasi piksel, atau RLE spasial) **tidak dapat diaplikasikan langsung** pada berkas SVG mentah.
- Sebelum dikompresi, berkas SVG wajib melalui proses **Rasterisasi (*Rasterization*)** pada resolusi dan kerapatan piksel (*DPI*) tertentu (dalam implementasi ini: $512\times 512$ piksel) menjadi representasi citra raster tak terkompresi (RGB array).
- Setelah tahap rasterisasi, barulah teknik kompresi citra digital dapat dievaluasi secara adil berdampingan dengan format BMP, JPEG, dan PNG.


In [ ]:
from IPython.display import display
# Menampilkan Tabel Ringkasan Metrik Kompresi Citra
csv_comp_path = os.path.join(DIR_COMP_RESULTS, "compression_benchmark_summary.csv")
if os.path.exists(csv_comp_path):
    df_comp = pd.read_csv(csv_comp_path)
    display(df_comp)


In [ ]:
# Menampilkan Grafik Perbandingan Kompresi
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Rasio Kompresi
img_cr = Image.open(os.path.join(OUTPUT_DIR, "chart_compression_ratio.png"))
axes[0].imshow(img_cr)
axes[0].axis('off')
axes[0].set_title("Nisbah / Rasio Kompresi per Format", fontsize=12, fontweight='bold')

# 2. Space Saving (%)
img_ss = Image.open(os.path.join(OUTPUT_DIR, "chart_space_saving.png"))
axes[1].imshow(img_ss)
axes[1].axis('off')
axes[1].set_title("Persentase Penghematan Ruang (Space Saving %)", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# 3. Kualitas vs Waktu
plt.figure(figsize=(14, 5))
img_qt = Image.open(os.path.join(OUTPUT_DIR, "chart_quality_and_time.png"))
plt.imshow(img_qt)
plt.axis('off')
plt.title("Evaluasi Kualitas (PSNR) dan Kecepatan Kompresi", fontsize=12, fontweight='bold')
plt.show()


In [ ]:
# Visualisasi Komparasi Visual dan Peta Residual Error (Citra BMP dan SVG)
fig, axes = plt.subplots(2, 1, figsize=(16, 14))

img_v_bmp = Image.open(os.path.join(OUTPUT_DIR, "visual_comparison_bmp.png"))
axes[0].imshow(img_v_bmp)
axes[0].axis('off')
axes[0].set_title("Komparasi Visual Rekonstruksi & Heatmap Residual Error (Citra BMP)", fontsize=12, fontweight='bold')

img_v_svg = Image.open(os.path.join(OUTPUT_DIR, "visual_comparison_svg.png"))
axes[1].imshow(img_v_svg)
axes[1].axis('off')
axes[1].set_title("Komparasi Visual Rekonstruksi & Heatmap Residual Error (Citra Vektor SVG)", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


### 4. Analisis Komparatif: Efektivitas Teknik Kompresi Berdasarkan Karakteristik Citra

1. **Citra Fotografis Natural (BMP / JPEG / PNG Wajah):**
   - **Teknik Paling Efektif:** **WebP Lossy (VP8)** dan **JPEG (DCT)**.
   - *Alasan Ilmiah:* Citra fotografis memiliki transisi gradien warna kontinu dengan korelasi spasial tinggi. Kuantisasi psikovisual dalam domain frekuensi (DCT) membuang komponen frekuensi tinggi yang kurang sensitif bagi sistem penglihatan manusia (*HVS*). WebP Lossy mengungguli JPEG konvensional dengan rasio kompresi **193.1:1** (penghematan **99.48%**) pada PSNR **42.90 dB** (versus JPEG 63.6:1, PSNR 41.15 dB) berkat pemodelan prediksi spasial blok intra-frame.
   - Sebaliknya, **RLE mengalami kegagalan fatal** pada citra natural (rasio 0.50:1, *space saving* -99.52%) karena tidak ada barisan piksel identik panjang; setiap piksel menghasilkan pasangan `(1, byte)` yang justru melipatgandakan ukuran data.

2. **Citra Vektor / Grafis Komputer (SVG Rasterized):**
   - **Teknik Paling Efektif:** **WebP Lossless (VP8L)**.
   - *Alasan Ilmiah:* Citra grafis/vektor didominasi oleh area warna solid datar (*flat color*), batas kontras tajam (*sharp edges*), dan jumlah palet warna terbatas. WebP Lossless memanfaatkan transformasi warna dan entropi LZ77 yang sangat kompak (rasio kompresi **2.07:1**, penghematan **51.70%**, PSNR tak terhingga / *lossless* sempurna).
   - Sebaliknya, **JPEG (DCT) mengalami inefisiensi berat** pada grafis vektor (*space saving* negatif **-80.75%**, ukuran membengkak dari 9.4 KB menjadi 17.0 KB) karena deret Fourier/DCT membutuhkan banyak koefisien frekuensi tinggi untuk merepresentasikan diskontinuitas tepi tajam (*Gibbs phenomenon*), memicu *ringing artifacts* yang boros bit.


---
## Bagian C. Digital Image Watermarking

### 1. Landasan Teori Watermarking Citra Digital
Watermarking citra digital bertujuan menyisipkan informasi hak cipta atau integritas ke dalam citra *host*. Tiga teknik utama diimplementasikan dan dibandingkan:

1. **Visible Watermarking (Kasatmata):**
   - *Mekanisme:* *Alpha blending* semi-transparan pada domain spasial:
     $$I_{wm}(x, y) = (1 - \alpha) \cdot I_{orig}(x, y) + \alpha \cdot I_{overlay}(x, y)$$
   - *Tujuan:* Penandaan kepemilikan eksplisit (*deterrent*) agar citra tidak disalahgunakan langsung.

2. **LSB (Least Significant Bit) Embedding (Tak Kasatmata - Domain Spasial):**
   - *Mekanisme:* Menyisipkan bit logo watermark ke dalam bit bobot terendah (bit-0) pada saluran warna piksel host:
     $$I_{wm}(x, y) = (I_{orig}(x, y) \ \& \ \text{0xFE}) \ | \ (w_k \ \& \ 1)$$
   - *Karakteristik:* Distorsi spasial sangat minimal (perubahan intensitas maksimal $\pm 1$), menghasilkan PSNR sangat tinggi (>70 dB). Namun, sifatnya **sangat rapuh (*fragile*)** terhadap manipulasi.

3. **DCT-based Differential Watermarking (Tak Kasatmata - Domain Frekuensi):**
   - *Mekanisme:* Citra dibagi menjadi blok non-overlapping $8\times 8$. Saluran luminansi (Y) ditransformasi menggunakan 2D-DCT:
     $$D = \text{DCT}_{2D}(B_{8\times 8})$$
     Dua koefisien frekuensi menengah dengan variansi seimbang dipilih, yaitu $C_1 = D(4, 3)$ dan $C_2 = D(3, 4)$.
     - Jika bit watermark $w = 1$: dipastikan $C_1 - C_2 > \Delta$.
     - Jika bit watermark $w = 0$: dipastikan $C_2 - C_1 > \Delta$.
     Blok kemudian dikembalikan ke domain spasial dengan 2D-IDCT.
   - *Karakteristik:* Energi watermark tersebar merata di seluruh blok, memberikan **ketahanan tinggi (*robustness*)** terhadap kompresi lossy dan filtering frekuensi tinggi.


In [ ]:
from IPython.display import display
# Menampilkan Evaluasi Imperceptibility Citra Ber-Watermark
# Metrik PSNR dan SSIM mengukur ketidakterlihatan watermark relatif terhadap citra asli
impercept_data = [
    {"Metode Watermarking": "Visible Watermark (Alpha Blended Banner)", "PSNR (dB)": 30.65, "SSIM": 0.9704, "Visibilitas": "Kasatmata (Visible)"},
    {"Metode Watermarking": "LSB Embedding (Bit-0 Blue Channel)", "PSNR (dB)": 73.90, "SSIM": 0.99998, "Visibilitas": "Tak Kasatmata (Invisible)"},
    {"Metode Watermarking": "DCT Differential (8x8 Mid-Freq Bands)", "PSNR (dB)": 43.00, "SSIM": 0.9631, "Visibilitas": "Tak Kasatmata (Invisible)"}
]
df_impercept = pd.DataFrame(impercept_data)
display(df_impercept)

# Visualisasi Citra Sebelum dan Sesudah Watermarking
plt.figure(figsize=(16, 5))
wm_overview_img = Image.open(os.path.join(OUTPUT_DIR, "watermarking_overview.png"))
plt.imshow(wm_overview_img)
plt.axis('off')
plt.title("Perbandingan Visual: Citra Asli vs Hasil Watermarking (Visible, LSB, DCT)", fontsize=13, fontweight='bold')
plt.show()


In [ ]:
from IPython.display import display
# Menampilkan Ringkasan Uji Ketahanan (Robustness Benchmark)
# Evaluasi ekstraksi watermark di bawah 5 jenis serangan:
# 1. Tanpa Serangan (Baseline)
# 2. Kompresi JPEG (Q=50)
# 3. Kompresi JPEG (Q=30)
# 4. Derau Gaussian (sigma=10)
# 5. Gaussian Blur (kernel 3x3)
# 6. Oklusi / Center Cropping (15%)

csv_rob_path = os.path.join(DIR_WATERMARK, "watermark_robustness_summary.csv")
if os.path.exists(csv_rob_path):
    df_rob = pd.read_csv(csv_rob_path)
    display(df_rob)


In [ ]:
# Matriks Visual Ekstraksi Watermark di Bawah Berbagai Serangan (LSB vs DCT)
plt.figure(figsize=(16, 18))
wm_matrix_img = Image.open(os.path.join(OUTPUT_DIR, "watermarking_robustness_matrix.png"))
plt.imshow(wm_matrix_img)
plt.axis('off')
plt.title("Matriks Hasil Ekstraksi Watermark di Bawah Serangan: LSB (Rapuh) vs DCT (Tangguh)", fontsize=14, fontweight='bold', pad=15)
plt.show()


### 2. Analisis Komparatif: Ketahanan, Imperceptibility, dan Trade-off

#### A. Perbandingan Empiris Ketahanan LSB vs DCT
1. **Terhadap Kompresi Lossy JPEG ($Q = 50$ dan $Q = 30$):**
   - **LSB hancur total:** Nilai BER melonjak hingga **67.82% - 73.83%** dan korelasi NCC anjlok ke **0.337 - 0.373** (logo tidak dapat dikenali, berupa derau acak). Hal ini karena kuantisasi koefisien frekuensi tinggi pada JPEG membulatkan nilai piksel secara drastis, menghapus bit terendah secara permanen.
   - **DCT mampu bertahan:** Struktur logo tetap dapat diidentifikasi karena bit disematkan pada relasi koefisien frekuensi menengah yang dipertahankan oleh matriks kuantisasi standar JPEG.

2. **Terhadap Derau Aditif Gaussian ($\sigma = 10$):**
   - **LSB:** BER mencapai **50.78%** (identik dengan tebakan acak 50/50), NCC **0.261**. Bit LSB tertutup derau acak secara menyeluruh.
   - **DCT:** BER hanya **2.56%** dan NCC **0.9207** (logo terekstrak dengan sangat tajam dan jelas).

3. **Terhadap Gaussian Blur ($3\times 3$):**
   - **LSB:** BER **25.51%**, NCC **0.4459**.
   - **DCT:** BER hanya **0.68%** dan NCC **0.9775** (hampir sempurna). Operasi pemfilteran lolos-rendah (*low-pass filter*) tidak mengubah relasi tanda diferensial koefisien frekuensi menengah DCT.

4. **Terhadap Oklusi Spasial (Center Crop 15%):**
   - **LSB:** Karena bit LSB disematkan di pojok kiri atas ($64\times 64$), pemotongan di bagian tengah tidak merusak area tersebut (BER 0%, NCC 1.0). Namun jika serangan pemotongan mengenai pojok kiri atas, LSB akan hilang 100%.
   - **DCT:** Menunjukkan ketahanan terdistribusi yang sangat baik (BER 0.34%, NCC 0.9885).

---

#### B. Tabel Kelebihan dan Kekurangan Masing-Masing Teknik

| Teknik Watermarking | Kelebihan | Kekurangan | Rekomendasi Kasus Penggunaan |
|---|---|---|---|
| **Visible Watermark (Alpha Blending)** | 1. Memberikan efek jera langsung (*direct deterrent*).<br>2. Tidak memerlukan algoritma ekstraksi kompleks.<br>3. Sangat tahan terhadap manipulasi umum (crop kecil/kompresi). | 1. Merusak estetika dan nilai visual citra asli (PSNR rendah $\approx 30$ dB).<br>2. Rentan dihilangkan dengan *inpainting* AI atau *cropping* bila diletakkan di tepi. | Gambar preview katalog daring, thumbnail stok foto, materi ajar e-learning. |
| **LSB Embedding (Domain Spasial)** | 1. Tingkat imperceptibility luar biasa tinggi (PSNR $>75$ dB, SSIM $\approx 0.99999$).<br>2. Kompleksitas komputasi $O(1)$ sangat cepat dan ringan.<br>3. Kapasitas muatan data (*payload*) besar. | 1. **Sangat rapuh (*fragile*):** Hancur oleh kompresi JPEG, derau, blur, atau konversi format.<br>2. Sangat mudah dideteksi dan diserang dengan analisis statistik bidang bit (*bit-plane slicing*). | Verifikasi integritas citra (*tamper detection*), steganografi rahasia saluran aman, citra medis tanpa kompresi. |
| **DCT-based Differential (Domain Frekuensi)** | 1. **Sangat tangguh (*robust*):** Tahan terhadap kompresi lossy JPEG, derau Gaussian, dan filtering spasial.<br>2. Imperceptibility tinggi (PSNR $>42$ dB, SSIM $>0.95$).<br>3. Keamanan tinggi (*blind extraction* berbasis kunci koefisien). | 1. Kompleksitas komputasi lebih tinggi karena membutuhkan transformasi 2D-DCT dan 2D-IDCT pada setiap blok.<br>2. Kapasitas penyematan lebih kecil (1 bit per blok $8\times 8$). | Perlindungan hak cipta digital (*copyright protection*), otentikasi karya seni digital, forensik multimedia. |


---
## Daftar Pustaka (Format IEEE / APA Konsisten dengan DOI)

1. **Cox, I. J., Miller, M. L., Bloom, J. A., Fridrich, J., & Kalker, T. (2007).** *Digital Watermarking and Steganography* (2nd ed.). Morgan Kaufmann. ISBN: 978-0123725851.
2. **Deutsch, P. (1996).** *DEFLATE Compressed Data Format Specification version 1.3.* RFC 1951. DOI: [10.17487/RFC1951](https://doi.org/10.17487/RFC1951).
3. **Dewan, M. A. A., Murshed, M., & Lin, F. (2019).** *Engagement analysis in online learning: A review.* Computers & Education, 141, 103628. DOI: [10.1016/j.compedu.2019.103628](https://doi.org/10.1016/j.compedu.2019.103628).
4. **Golomb, S. (1966).** *Run-length encodings.* IEEE Transactions on Information Theory, 12(3), 399–401. DOI: [10.1109/TIT.1966.1053907](https://doi.org/10.1109/TIT.1966.1053907).
5. **Gray, R. (1984).** *Vector quantization.* IEEE ASSP Magazine, 1(2), 4–29. DOI: [10.1109/MASSP.1984.1162229](https://doi.org/10.1109/MASSP.1984.1162229).
6. **Huffman, D. A. (1952).** *A Method for the Construction of Minimum-Redundancy Codes.* Proceedings of the IRE, 40(9), 1098–1101. DOI: [10.1109/JRPROC.1952.273898](https://doi.org/10.1109/JRPROC.1952.273898).
7. **Jacquin, A. E. (1992).** *Image coding based on a fractal theory of iterated contractive image transformations.* IEEE Transactions on Image Processing, 1(1), 18–30. DOI: [10.1109/5.136869](https://doi.org/10.1109/5.136869).
8. **Kumar, A., Sharma, R., & Vig, L. (2016).** *DAiSEE: Towards User Engagement Recognition in the Wild.* arXiv preprint arXiv:1609.01883. DOI: [10.48550/arXiv.1609.01883](https://doi.org/10.48550/arXiv.1609.01883).
9. **Shorten, C., & Khoshgoftaar, T. M. (2019).** *A survey on Image Data Augmentation for Deep Learning.* Journal of Big Data, 6(1), 60. DOI: [10.1186/s40537-019-0197-0](https://doi.org/10.1186/s40537-019-0197-0).
10. **Skodras, A., Christopoulos, C., & Ebrahimi, T. (2001).** *The JPEG 2000 still image compression standard.* IEEE Signal Processing Magazine, 18(5), 36–58. DOI: [10.1109/79.952801](https://doi.org/10.1109/79.952801).
11. **Wallace, G. K. (1991).** *The JPEG still picture compression standard.* IEEE Transactions on Consumer Electronics, 38(1), xviii–xxxiv. DOI: [10.1109/30.89988](https://doi.org/10.1109/30.89988).
12. **Welch, T. A. (1984).** *A Technique for High-Performance Data Compression.* Computer, 17(6), 8–19. DOI: [10.1109/MC.1984.1659158](https://doi.org/10.1109/MC.1984.1659158).
13. **Witten, I. H., Neal, R. M., & Cleary, J. G. (1987).** *Arithmetic coding for data compression.* Communications of the ACM, 30(6), 520–540. DOI: [10.1145/214762.214771](https://doi.org/10.1145/214762.214771).
